# Cut-out prototype

Runs `pipeline.py` step by step on one garment so you can see each stage and tune settings.
Outputs go to `output/` (git-ignored). **Clear outputs before committing** (nbstripout does this automatically once installed) — they embed photos.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import pipeline as P

s = P.Settings(mirror=True)      # mirror=True: photos were taken in a mirror
GARMENT, VIEW = "hoodie-black-zip", "front"

models = P.Models()
body = models.analyse(P.load_photo(P.find_photo(P.PHOTOS / "body", VIEW), s))
garment_raw = models.analyse(P.load_photo(P.find_photo(P.PHOTOS / "garments" / GARMENT, VIEW), s))

## 1. Pose and segmentation
Segmenter classes: 0 background, 1 hair, 2 body skin, 3 face skin, 4 clothes, 5 other.

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(16, 6))
for a, (title, img) in zip(ax, [
    ("body + landmarks", P.draw_landmarks(body.rgb, body.landmarks, (0, 255, 0), ids=range(33))),
    ("body classes", body.classes),
    ("garment + landmarks", P.draw_landmarks(garment_raw.rgb, garment_raw.landmarks, (255, 0, 255), ids=range(33))),
    ("garment classes", garment_raw.classes),
]):
    a.imshow(img, cmap="tab10", vmin=0, vmax=9) if img.ndim == 2 else a.imshow(img)
    a.set_title(title); a.axis("off")

## 2. Align garment photo to the body figure, then repose its limbs

In [ ]:
report = P.Report(GARMENT, VIEW)
garment, valid = P.align(garment_raw, body, s, report)
before = garment.rgb
# Bend the garment's arms/legs onto the body figure's (limbs turning > 40 deg are left as captured).
garment, valid = P.repose_analysis(garment, valid, body.landmarks, report)
print(report)
fig, ax = plt.subplots(1, 2, figsize=(14, 8))
for a, (title, img) in zip(ax, [("before repose", before), ("after repose", garment.rgb)]):
    a.imshow((0.5 * body.rgb + 0.5 * img).astype("uint8")); a.set_title(title + ": 50/50 blend with body, ghosting = misalignment"); a.axis("off")

## 3. Cut out
Try changing `s.change_threshold` (colour difference that counts as garment) and re-run.

In [ ]:
rgba, reason, change = P.cut_out(garment, valid, body, s, report)
sheet = P.comparison_sheet(body, garment, rgba, reason, change, report)
plt.figure(figsize=(24, 7)); plt.imshow(sheet[..., ::-1]); plt.axis("off")
print(report.removed_px, report.warnings)